- 필요한 라이브러리를 불러오겠습니다.

In [1]:
from ucimlrepo import fetch_ucirepo

import pandas as pd
import numpy as np


- 필요한 데이터셋을 불러오겠습니다. (위치는 ../data 폴더입니다.)
  - 데이터셋 출처 : [UCI Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease)

In [2]:
# fetch dataset 
heart_disease = fetch_ucirepo(id=45) 
  
# data (as pandas dataframes) 
X = heart_disease.data.features 
y = heart_disease.data.targets 
  
# metadata 
# print(heart_disease.metadata) 
  
# variable information 
# print(heart_disease.variables) 

# feature_df = pd.read_table(X)

X.to_csv('../data/heart_disease_feature.csv')
y.to_csv('../data/heart_disease_target.csv')


In [3]:
feature_file_path = "..\\data\\heart_disease_feature.csv"
target_file_path = "..\\data\\heart_disease_target.csv"

feature_data = pd.read_csv(feature_file_path)
target_data = pd.read_csv(target_file_path)

 - [UCI Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease) 을 참고하여 데이터셋에 관해 정리해보겠습니다.
   - 우선 여러 Attribute가 있지만, 그 중에서도 해당 데이터셋에는 전부가 아닌 일부분만 사용되었습니다.
   - 그 중 num Attribute가 바로 심장병에 대한 값, 즉 target Attribute에 해당합니다.
   - 이에 따라 나머지 Attribute(age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal)이 자연스럽게 Feature Attribute라는 것으로 자명됩니다.

---

- Feature 데이터셋의 column 값들을 간단하게 확인하겠습니다.

In [4]:
print(feature_data.head())

   Unnamed: 0  age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  \
0           0   63    1   1       145   233    1        2      150      0   
1           1   67    1   4       160   286    0        2      108      1   
2           2   67    1   4       120   229    0        2      129      1   
3           3   37    1   3       130   250    0        0      187      0   
4           4   41    0   2       130   204    0        2      172      0   

   oldpeak  slope   ca  thal  
0      2.3      3  0.0   6.0  
1      1.5      2  3.0   3.0  
2      2.6      2  2.0   7.0  
3      3.5      3  0.0   3.0  
4      1.4      1  0.0   3.0  


- 이번엔 Target 데이터 셋입니다.

In [5]:
print(target_data.head())

   Unnamed: 0  num
0           0    0
1           1    2
2           2    1
3           3    0
4           4    0


- 데이터의 전체적인 구조에 대한 요약과 column들을 확인해보겠습니다.
  - 아래 결과를 통해 대부분은 `int64`의 타입을 가지고 있지만, oldpeak, ca, thal은 `float64`의 타입을 가진다는 것을 알 수 있습니다.

In [6]:
print(feature_data.info())
print(feature_data.describe())

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  303 non-null    int64  
 1   age         303 non-null    int64  
 2   sex         303 non-null    int64  
 3   cp          303 non-null    int64  
 4   trestbps    303 non-null    int64  
 5   chol        303 non-null    int64  
 6   fbs         303 non-null    int64  
 7   restecg     303 non-null    int64  
 8   thalach     303 non-null    int64  
 9   exang       303 non-null    int64  
 10  oldpeak     303 non-null    float64
 11  slope       303 non-null    int64  
 12  ca          299 non-null    float64
 13  thal        301 non-null    float64
dtypes: float64(3), int64(11)
memory usage: 33.3 KB
None
       Unnamed: 0         age         sex          cp    trestbps        chol  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  303.000000   
mean   151.000000   54.438944  

- 이번엔 target 데이터 셋입니다.
  - 전체가 `int64` 타입으로 이루어져 있습니다.

In [7]:
print(target_data.info())
print(target_data.describe())

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  303 non-null    int64
 1   num         303 non-null    int64
dtypes: int64(2)
memory usage: 4.9 KB
None
       Unnamed: 0         num
count  303.000000  303.000000
mean   151.000000    0.937294
std     87.612784    1.228536
min      0.000000    0.000000
25%     75.500000    0.000000
50%    151.000000    0.000000
75%    226.500000    2.000000
max    302.000000    4.000000


---

- 클래스 분포를 확인해보겠습니다.
  - 아래 코드로 확인해본 결과 결과에 문제점이 나타났는데, 우선 심장병의 정도에 따라서 0, 1, 2 등으로 나뉘어서 클래스 분포를 알 수가 없습니다. (여기서 구하려는건 심장병의 유무에 따라서 클래스의 분포를 구하려는 것입니다.)

In [13]:
print(target_data['num'].value_counts(normalize=True))

num
0    0.541254
1    0.181518
2    0.118812
3    0.115512
4    0.042904
Name: proportion, dtype: float64


- 따라서 코드를 이진화 한 뒤 다시 구해보겠습니다.

In [9]:
target_data['binary_num'] = pd.cut(target_data['num'], 
                                   bins=[-np.inf, 0, np.inf], 
                                   labels=[0, 1])
print(target_data)

     Unnamed: 0  num binary_num
0             0    0          0
1             1    2          1
2             2    1          1
3             3    0          0
4             4    0          0
..          ...  ...        ...
298         298    1          1
299         299    2          1
300         300    3          1
301         301    1          1
302         302    0          0

[303 rows x 3 columns]


In [12]:
print(target_data['binary_num'].value_counts(normalize=True))

binary_num
0    0.541254
1    0.458746
Name: proportion, dtype: float64


- 결과적으로 절반씩 분포되어있지만, 0(심장병 없음) 클래스가 약간 더 많은 것으로 나타났습니다.
- 이 케이스가 다행인 점은 한쪽에 엄청 편향되어있지 않다는 점입니다.
  - 만일 클래스가 너무 한쪽으로 분포되어있으면, 해당 클래스 내에서 정밀도, 재현율, F1 점수 등을 사용해야 했었습니다.

---